<a href="https://colab.research.google.com/github/ASpacesys/camera-pose-estimation-from-landmarks/blob/main/Camera_Pose_Estimation_from_Landmarks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Finding an Observer's Position from a Photograph

This notebook aims to find the exact position of any photo given identifiable landmarks with known coordinates. It gives two solutions: a range of possible locations to a single line, and an attempt to find the exact latitude, longitude, and altitude of the observer. In my own experiences with the project, the line of locations tends to be within 200 feet/mile from your landmarks. Note that this is only an upper bound for the error in the worst cases.

**For reliable results:**
- Use as many landmarks as possible for searches. Lower values contain much more variance and likely won't yield as accurate of solutions.
- Try to pick points that span a large depth field and field of view. The more spread out the points, the more accurate the search will be.

**Notes:**
 -
 - As of now, the line of possible locations uses a reprojection error search. It may be replaced in the future with a method which doesn't require camera intrinsics, although the current version is relatively accurate.
   - Known issue: The line of possible locations can be flipped along the Z-axis of the ENU origin point.  
 - The earth's curvature is also unaccounted for, although this error is much less significant than the above. ENU coordinates may be replaced with ECEF and geodetics in the future.

In [ ]:
#@title Install Dependencies

!pip install pymap3d opencv-python -q

In [ ]:
#@title Enter Google Maps API Key

# Used for interactive world coordinate entry and Google Maps Visualization. You can get a free personal Maps Demo API Key.
GOOGLE_MAPS_API_KEY = ""

In [ ]:
#@title Input Pixel Coordinates + Build Arrays
import sys, io, base64, numpy as np
IN_COLAB = 'google.colab' in sys.modules
image_points_px = []

SENSOR_W_MM = 36.0   # full-frame reference width; used with 35mm-equivalent focal lengths

def mm_to_px(f_mm, sensor_w_mm=SENSOR_W_MM):
    return np.asarray(f_mm) / sensor_w_mm * image_width_px

def px_to_mm(f_px, sensor_w_mm=SENSOR_W_MM):
    return np.asarray(f_px) / image_width_px * sensor_w_mm


def build_arrays():
    """Run after clicking 'Done' in the widget. Sets pixel_coords, K, dist_coeffs."""
    global pixel_coords, K, dist_coeffs, cx, cy, fx, fy
    if not image_points_px:
        return print("No points received yet. Click 'Done' in the widget first.")
    pixel_coords = np.array(image_points_px, dtype=np.float64)
    cx, cy = image_width_px / 2, image_height_px / 2   # optical center = image center
    fx = globals().get('fx', image_width_px)           # rough focal length guess
    fy = globals().get('fy', image_width_px)
    K = np.array([[fx, 0, cx], [0, fy, cy], [0, 0, 1]], dtype=np.float64)
    dist_coeffs = np.zeros(5)
    print(pixel_coords)

if not IN_COLAB:
    print("Use manual entry instead, or run this notebook in Colab.")
else:
    from google.colab import files, output
    from PIL import Image
    from IPython.display import HTML, display

    print("Choose your image file:")
    up = files.upload()
    name = next(iter(up))
    img = Image.open(io.BytesIO(up[name])).convert('RGB')
    image_width_px, image_height_px = img.size
    print(f"Loaded '{name}': {image_width_px} x {image_height_px} px")

    buf = io.BytesIO(); img.save(buf, format='JPEG', quality=90)
    b64 = base64.b64encode(buf.getvalue()).decode()
    scale = min(1.0, 1000 / image_width_px)

    def set_image_points(points):
        global image_points_px
        image_points_px = points
        build_arrays()          # builds pixel_coords/K automatically on "Done"

    output.register_callback('notebook.set_image_points', set_image_points)

    display(HTML(f'''
<div>
  <div style="margin-bottom:6px;">
    <button id="u">Undo last point</button>
    <button id="c">Clear all</button>
    <button id="d" style="font-weight:bold;">Done</button>
    <span id="n" style="margin-left:10px;"></span>
  </div>
  <canvas id="cv" width="{int(image_width_px*scale)}" height="{int(image_height_px*scale)}"
          style="border:1px solid #ccc;cursor:crosshair;"></canvas>
</div>
<script>
(() => {{
  const s = {scale}, cv = document.getElementById('cv'), ctx = cv.getContext('2d');
  const img = new Image(); let pts = [];
  const redraw = () => {{
    ctx.clearRect(0, 0, cv.width, cv.height);
    ctx.drawImage(img, 0, 0, cv.width, cv.height);
    pts.forEach((p, i) => {{
      ctx.beginPath(); ctx.arc(p[0]*s, p[1]*s, 6, 0, 6.2832);
      ctx.fillStyle = 'red'; ctx.fill();
      ctx.fillStyle = 'white'; ctx.font = 'bold 14px sans-serif';
      ctx.fillText(i+1, p[0]*s+8, p[1]*s-8);
    }});
    document.getElementById('n').innerText = pts.length + ' point(s) clicked';
  }};
  img.onload = redraw;
  img.src = 'data:image/jpeg;base64,{b64}';
  cv.onclick = e => {{
    const r = cv.getBoundingClientRect();
    pts.push([(e.clientX-r.left)/s, (e.clientY-r.top)/s]); redraw();
  }};
  u.onclick = () => {{ pts.pop(); redraw(); }};
  c.onclick = () => {{ pts = []; redraw(); }};
  d.onclick = () => {{
    google.colab.kernel.invokeFunction('notebook.set_image_points', [pts], {{}});
    document.getElementById('n').innerText = pts.length + ' point(s) captured!';
  }};
}})();
</script>'''))

# Interactive World Coordinate Entry
Instead of finding latitude, longitude, and altitude through other resources, the next two cells give the option to click landmarks on Google Maps. This is most useful for aerial images. For ground or near-ground images, skip ahead to input coordinates manually.

In [ ]:
#@title Input World Coordinates (requires API key) + Build World Array
import sys, requests, numpy as np, pymap3d as pm
IN_COLAB = 'google.colab' in sys.modules

map_center_lat, map_center_lon = 37.70819703505958, -122.29786392135888
map_zoom = 11
map_points_latlng = []

def build_world(lla):
    """lla: list of (lat, lon, alt_msl_m). Sets landmarks_lla and world_points (ENU, meters)."""
    global landmarks_lla, world_points
    n_px = len(pixel_coords) if 'pixel_coords' in globals() else None
    if n_px is not None and n_px != len(lla):
        return print(f"\n\u26a0\ufe0f MISMATCH: {n_px} image points vs {len(lla)} landmarks. "
                     "Click the same number of points, in the same order, on both widgets.")
    landmarks_lla = list(lla)
    lat0, lon0, alt0 = landmarks_lla[0]
    world_points = np.array([pm.geodetic2enu(la, lo, al, lat0, lon0, alt0)
                             for la, lo, al in landmarks_lla], dtype=np.float64)
    print(f"OK -- {len(landmarks_lla)} landmarks" + (f" match {n_px} image points." if n_px else "."))
    for i, ((la, lo, al), (e, n, u)) in enumerate(zip(landmarks_lla, world_points), 1):
        print(f"  {i}: lat/lon/alt = {la:.6f}, {lo:.6f}, {al:.1f} m MSL  ->  ENU = {e:.2f}, {n:.2f}, {u:.2f} m")

def get_msl_elevations(pts):
    r = requests.get("https://api.open-meteo.com/v1/elevation", params={
        "latitude": ",".join(str(p['lat']) for p in pts),
        "longitude": ",".join(str(p['lng']) for p in pts)}).json()
    if 'elevation' not in r:
        raise RuntimeError(f"Open-Meteo API error: {r}")
    return r['elevation']

if not IN_COLAB:
    print("This interactive widget requires Google Colab. Use the manual cell below.")
else:
    from google.colab import output
    from IPython.display import HTML, display

    def set_map_points(points):
        global map_points_latlng
        map_points_latlng = points
        print(f"Received {len(points)} map point(s) from the widget.")
        if points:
            build_world([(p['lat'], p['lng'], e) for p, e in zip(points, get_msl_elevations(points))])

    output.register_callback('notebook.set_map_points', set_map_points)

    display(HTML(f'''
<div>
  <div style="margin-bottom:6px;">
    <button id="u">Undo last point</button>
    <button id="c">Clear all</button>
    <button id="d" style="font-weight:bold;">Done</button>
    <span id="n" style="margin-left:10px;"></span>
  </div>
  <div id="map" style="width:100%;height:600px;border:1px solid #ccc;"></div>
</div>
<script>
(() => {{
  let pts = [], markers = [], map;
  const label = t => document.getElementById('n').innerText = t;
  const relabel = () => {{
    markers.forEach((m, i) => m.setLabel(String(i+1)));
    label(pts.length + ' point(s) clicked');
  }};
  window.pnpInitMap = () => {{
    map = new google.maps.Map(document.getElementById('map'),
      {{center: {{lat: {map_center_lat}, lng: {map_center_lon}}}, zoom: {map_zoom}, mapTypeId: 'hybrid'}});
    map.addListener('click', e => {{
      const p = {{lat: e.latLng.lat(), lng: e.latLng.lng()}};
      pts.push(p);
      markers.push(new google.maps.Marker({{position: p, map}}));
      relabel();
    }});
  }};
  u.onclick = () => {{ pts.pop(); const m = markers.pop(); if (m) m.setMap(null); relabel(); }};
  c.onclick = () => {{ pts = []; markers.forEach(m => m.setMap(null)); markers = []; relabel(); }};
  d.onclick = () => {{
    google.colab.kernel.invokeFunction('notebook.set_map_points', [pts], {{}});
    label(pts.length + ' point(s) captured!');
  }};
}})();
</script>
<script src="https://maps.googleapis.com/maps/api/js?key={GOOGLE_MAPS_API_KEY}&callback=pnpInitMap" async defer></script>'''))

# Manual World Coordinate Entry
For ground or near-ground images, world coordinates should be input manually. This allows you to use points such as faces of buildings as landmarks. Points should be input in the format of (latitude, longitude, altitude (MSL, meters)).

In [ ]:
#@title Manual World Coordinate Input
# (latitude, longitude, altitude MSL in meters)
build_world([
    (42.28419675439698, -83.74130034799083, 270),
    (42.284242890354214, -83.74120311791316, 270),
    (42.28428654585253, -83.74129699522953, 270),
    (42.28591084824175, -83.74115475468355, 268),
    (42.28597781795939, -83.74113463811577, 268),
])

# Solving for possible positions
Zooming in while moving backwards or zooming out while moving forwards ("dolly zooms") can make objects in the frame appear the same size. While this makes identifying the exact location harder, it is easy to determine the line in 3D space which the observer might lie on. Note that this line does not correspond to the direction in which the observer is looking.

In [ ]:
#@title Camera-Location Reprojection Error Heatmap (30 mi around landmark 1)
import time, numpy as np, pymap3d as pm, folium
import matplotlib.cm as cm
from matplotlib.colors import Normalize, to_hex
from branca.colormap import LinearColormap

# ---- settings ----
RADIUS_MI     = 30
RES_M         = 500                                   # coarser, since we're adding two more dimensions
CAM_HEIGHTS_M = np.linspace(100, 12000, 10)          # airplane-scale heights
FOCAL_MM      = np.linspace(18, 200, 15)      # 35mm-equiv focal lengths to search (mm)
FOCAL_PX      = mm_to_px(FOCAL_MM)
OPACITY       = 0.6
CHUNK         = 40_000

# ---- setup ----
lat0, lon0, alt0 = landmarks_lla[0]
radius_m = RADIUS_MI * 1609.344
assert len(pixel_coords) == len(world_points), "pixel/world point counts differ"
n = len(world_points)

# Unit bearing rays in the camera frame (OpenCV convention: x right, y down, z forward)
rays = (np.linalg.inv(K) @ np.c_[pixel_coords, np.ones(n)].T).T
rays /= np.linalg.norm(rays, axis=1, keepdims=True)
fx_, fy_, cx_, cy_ = K[0, 0], K[1, 1], K[0, 2], K[1, 2]

# ---- grid in local ENU (landmark 1 = origin), masked to a circle ----
axis = np.arange(-radius_m, radius_m + RES_M, RES_M)
E, N = np.meshgrid(axis, axis)                      # N rows ascend (south -> north)
inside = (E**2 + N**2) <= radius_m**2
cells = np.c_[E[inside], N[inside]]
print(f"Grid: {len(axis)}x{len(axis)}, {len(cells):,} cells in radius, {len(CAM_HEIGHTS_M)} heights")

def reproj_rms(cam):
    """cam: (c,3) camera positions in ENU -> (c,) RMS pixel error (nan if any point is behind camera)."""
    v = world_points[None, :, :] - cam[:, None, :]                  # (c,n,3)
    d = v / np.linalg.norm(v, axis=2, keepdims=True)
    H = np.einsum('cni,nj->cij', d, rays)                           # sum d_i r_i^T
    U, _, Vt = np.linalg.svd(H)
    V = Vt.transpose(0, 2, 1)
    s = np.sign(np.linalg.det(V @ U.transpose(0, 2, 1)))            # enforce a proper rotation
    D = np.ones((len(cam), 3)); D[:, 2] = s
    R = (V * D[:, None, :]) @ U.transpose(0, 2, 1)                  # world -> camera, best fit
    Rv = np.einsum('cij,cnj->cni', R, v)
    z = Rv[..., 2]
    with np.errstate(divide='ignore', invalid='ignore'):
        u = fx_ * Rv[..., 0] / z + cx_
        w = fy_ * Rv[..., 1] / z + cy_
        err = np.sqrt(np.mean((u - pixel_coords[:, 0])**2 + (w - pixel_coords[:, 1])**2, axis=1))
    err[(z <= 1e-6).any(axis=1)] = np.nan
    return err

# ---- evaluate ----
K0 = K.copy()
best = np.full(len(cells), np.inf); best_h = np.zeros(len(cells)); best_f = np.zeros(len(cells))
t0 = time.time()
for f_mm, f_px in zip(FOCAL_MM, FOCAL_PX):
    fx_ = fy_ = f_px
    Kf = K0.copy(); Kf[0, 0], Kf[1, 1] = f_px, f_px
    rays = (np.linalg.inv(Kf) @ np.c_[pixel_coords, np.ones(n)].T).T
    rays /= np.linalg.norm(rays, axis=1, keepdims=True)
    for h in CAM_HEIGHTS_M:
        for s0 in range(0, len(cells), CHUNK):
            sl = slice(s0, s0 + CHUNK)
            cam = np.c_[cells[sl], np.full(len(cells[sl]), float(h))]
            e = np.nan_to_num(reproj_rms(cam), nan=np.inf)
            better = e < best[sl]
            best[sl] = np.where(better, e, best[sl])
            best_h[sl] = np.where(better, h, best_h[sl])
            best_f[sl] = np.where(better, f_mm, best_f[sl])
best[~np.isfinite(best)] = np.nan
print(f"Done in {time.time()-t0:.1f}s")

grid = np.full(E.shape, np.nan); grid[inside] = best

# ---- best location ----
i = np.nanargmin(best)
be, bn = cells[i]
blat, blon, _ = pm.enu2geodetic(be, bn, best_h[i], lat0, lon0, alt0)
print(f"Lowest error: {best[i]:.2f} px at ({blat:.6f}, {blon:.6f}), ~{best_h[i]:.0f} m above landmark 1")

# ---- colorize: low error = red, high = blue (log scale, clipped) ----
logerr = np.log10(grid)
lo, hi = np.nanpercentile(logerr, [0, 95])
cmap = cm.get_cmap('RdYlBu')                                        # low -> red, high -> blue
rgba = cmap(Normalize(lo, hi, clip=True)(logerr))
rgba[np.isnan(grid)] = 0                                            # transparent outside circle / invalid
rgba = np.flipud(rgba)                                              # image row 0 = north

# ---- map ----
sw = pm.enu2geodetic(-radius_m - RES_M/2, -radius_m - RES_M/2, 0, lat0, lon0, alt0)[:2]
ne = pm.enu2geodetic( radius_m + RES_M/2,  radius_m + RES_M/2, 0, lat0, lon0, alt0)[:2]

m = folium.Map(location=[lat0, lon0], zoom_start=10, tiles=None)
folium.TileLayer('https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
                 attr='Esri World Imagery', name='Satellite').add_to(m)
folium.TileLayer('OpenStreetMap', name='Streets').add_to(m)
folium.raster_layers.ImageOverlay(rgba, bounds=[list(sw), list(ne)], opacity=OPACITY,
                                  name='Reprojection error').add_to(m)
for k, (la, lo_, _) in enumerate(landmarks_lla, 1):
    folium.CircleMarker([la, lo_], radius=4, color='black', fill=True, fill_color='white',
                        tooltip=f"Landmark {k}").add_to(m)
folium.Marker([blat, blon], tooltip=f"Best: {best[i]:.2f} px",
              icon=folium.Icon(color='green', icon='camera')).add_to(m)
legend_colors = [to_hex(cmap(x)) for x in np.linspace(0, 1, 8)]
LinearColormap(legend_colors, vmin=10**lo, vmax=10**hi,
               caption='RMS reprojection error (px, log scale)').add_to(m)
folium.TileLayer(
    'https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}',
    attr='Esri World Street Map',
    name='Streets',
).add_to(m)
m

In [ ]:
#@title Drag a box to zoom in: finer-resolution re-search
import time, json, io, base64, numpy as np, pymap3d as pm, matplotlib
from PIL import Image
from matplotlib.colors import Normalize, to_hex
from IPython.display import HTML, display
from google.colab import output

# ---- settings for the refined search ----
TARGET_CELLS  = 4_000                      # grid cells inside the box (resolution is derived from this)
MIN_RES_M     = 2                           # never go finer than this
CAM_HEIGHTS_M = np.linspace(100, 12000, 100)  # heights above landmark 1 (m)
FOCAL_MM      = np.linspace(18, 200, 15)      # 35mm-equiv focal lengths to search (mm)
FOCAL_PX      = mm_to_px(FOCAL_MM)
COLOR_PCT     = (0, 95)                     # percentile range mapped red -> blue
OPACITY       = 0.65
CHUNK         = 40_000

# ---- shared geometry (uses variables from the earlier cells) ----
K0 = K.copy(); cx_, cy_ = K0[0, 2], K0[1, 2]
lat0, lon0, alt0 = landmarks_lla[0]
n_pts = len(world_points)

def reproj_rms(cam, rays, fx, fy):
    """cam: (c,3) ENU positions -> (c,) RMS pixel error with best-fit orientation per camera."""
    v = world_points[None, :, :] - cam[:, None, :]
    d = v / np.linalg.norm(v, axis=2, keepdims=True)
    H = np.einsum('cni,nj->cij', d, rays)
    U, _, Vt = np.linalg.svd(H)
    V = Vt.transpose(0, 2, 1)
    s = np.sign(np.linalg.det(V @ U.transpose(0, 2, 1)))
    D = np.ones((len(cam), 3)); D[:, 2] = s
    R = (V * D[:, None, :]) @ U.transpose(0, 2, 1)
    Rv = np.einsum('cij,cnj->cni', R, v)
    z = Rv[..., 2]
    with np.errstate(divide='ignore', invalid='ignore'):
        u = fx * Rv[..., 0] / z + cx_
        w = fy * Rv[..., 1] / z + cy_
        err = np.sqrt(np.mean((u - pixel_coords[:, 0])**2 + (w - pixel_coords[:, 1])**2, axis=1))
    err[(z <= 1e-6).any(axis=1)] = np.nan
    return err

def search(e_lo, e_hi, n_lo, n_hi, res):
    es = np.arange(e_lo + res/2, e_hi, res)
    ns = np.arange(n_lo + res/2, n_hi, res)
    E, N = np.meshgrid(es, ns)
    cells = np.c_[E.ravel(), N.ravel()]
    best = np.full(len(cells), np.inf); bh = np.zeros(len(cells)); bf = np.zeros(len(cells))
    print(f"Grid {len(es)}x{len(ns)} = {len(cells):,} cells @ {res:.1f} m, "
          f"{len(FOCAL_MM)} focals x {len(CAM_HEIGHTS_M)} heights")
    t0 = time.time()
    for f_mm, f_px in zip(FOCAL_MM, FOCAL_PX):
        fx = fy = f_px
        Kf = K0.copy(); Kf[0, 0], Kf[1, 1] = f_px, f_px
        rays = (np.linalg.inv(Kf) @ np.c_[pixel_coords, np.ones(n_pts)].T).T
        rays /= np.linalg.norm(rays, axis=1, keepdims=True)
        for h in CAM_HEIGHTS_M:
            for s0 in range(0, len(cells), CHUNK):
                sl = slice(s0, s0 + CHUNK)
                cam = np.c_[cells[sl], np.full(len(cells[sl]), float(h))]
                e = np.nan_to_num(reproj_rms(cam, rays, fx, fy), nan=np.inf)
                better = e < best[sl]
                best[sl] = np.where(better, e, best[sl])
                bh[sl] = np.where(better, h, bh[sl]); bf[sl] = np.where(better, f_mm, bf[sl])
    best[~np.isfinite(best)] = np.nan
    print(f"Search finished in {time.time()-t0:.1f}s")
    shp = E.shape
    return es, ns, best.reshape(shp), bh.reshape(shp), bf.reshape(shp)

# ---- persistent view stack: [0] = original 30-mile view, [-1] = current ----
# Fingerprint of everything the original view depends on; if it changes (new photo, new
# landmarks, new focal/K), the cached stack is discarded and rebuilt from the current `grid`.
import hashlib
_fp = hashlib.md5(b''.join(np.ascontiguousarray(a, dtype=np.float64).tobytes()
                           for a in (pixel_coords, world_points, K, np.asarray(grid))) ).hexdigest()

if globals().get('view_stack_fp') != _fp or 'view_stack' not in globals():
    j = np.unravel_index(np.nanargmin(grid), grid.shape)     # grid/axis/RES_M come from the first heatmap cell
    view_stack = [dict(err=grid.copy(),
                       e_lo=axis[0] - RES_M/2, e_hi=axis[-1] + RES_M/2,
                       n_lo=axis[0] - RES_M/2, n_hi=axis[-1] + RES_M/2, res=RES_M,
                       best=dict(e=axis[j[1]], n=axis[j[0]], err=np.nanmin(grid), h=None, f=None))]
    view_stack_fp = _fp
    pending = None
    print("New inputs detected: view stack reset to the current original heatmap.")
if 'pending' not in globals():
    pending = None

def set_action(kind, payload=None):
    global pending
    pending = (kind, payload)
    print({'box': 'Box received', 'back': 'Back requested', 'reset': 'Reset requested'}[kind]
          + ". Re-run this cell to apply it.")
output.register_callback('notebook.set_action', set_action)

# ---- apply the pending action (one-shot) ----
if pending:
    kind, payload = pending
    pending = None
    if kind == 'reset':
        del view_stack[1:]
    elif kind == 'back':
        if len(view_stack) > 1: view_stack.pop()
    elif kind == 'box':
        s, w, nn, ee = payload
        (e1, n1), (e2, n2) = [pm.geodetic2enu(la, lo, alt0, lat0, lon0, alt0)[:2] for la, lo in [(s, w), (nn, ee)]]
        e_lo, e_hi = sorted((e1, e2)); n_lo, n_hi = sorted((n1, n2))
        res = max(MIN_RES_M, np.sqrt((e_hi - e_lo) * (n_hi - n_lo) / TARGET_CELLS))
        es, ns, err, bh, bf = search(e_lo, e_hi, n_lo, n_hi, res)
        k = np.unravel_index(np.nanargmin(err), err.shape)
        view_stack.append(dict(err=err, e_lo=e_lo, e_hi=e_lo + len(es) * res, n_lo=n_lo, n_hi=n_lo + len(ns) * res,
                               res=res, best=dict(e=es[k[1]], n=ns[k[0]], err=err[k], h=bh[k], f=bf[k]),
                               bh=bh, bf=bf))

view = view_stack[-1]          # downstream cells (line fit, 3D, KML) keep using `view`
print(f"Zoom level {len(view_stack) - 1}")

# ---- summary ----
b = view['best']
blat, blon, _ = pm.enu2geodetic(b['e'], b['n'], 0, lat0, lon0, alt0)
msg = f"View: {view['res']:.1f} m resolution | best {b['err']:.2f} px at ({blat:.6f}, {blon:.6f})"
if b['h'] is not None:
    msg += f" | focal {b['f']:.0f} mm (35mm-equiv, fx = {mm_to_px(b['f']):.0f} px)"
print(msg)

# ---- colorize ----
logerr = np.log10(view['err'])
lo, hi = np.nanpercentile(logerr, COLOR_PCT)
cmap = matplotlib.colormaps['RdYlBu']                       # low error = red, high = blue
rgba = cmap(Normalize(lo, hi, clip=True)(logerr))
rgba[np.isnan(logerr)] = 0
buf = io.BytesIO(); Image.fromarray((np.flipud(rgba) * 255).astype(np.uint8)).save(buf, format='PNG')
png_b64 = base64.b64encode(buf.getvalue()).decode()
sw = pm.enu2geodetic(view['e_lo'], view['n_lo'], 0, lat0, lon0, alt0)[:2]
ne = pm.enu2geodetic(view['e_hi'], view['n_hi'], 0, lat0, lon0, alt0)[:2]
grad = ", ".join(to_hex(cmap(x)) for x in np.linspace(0, 1, 8))

# ---- Leaflet map with a draggable-box tool ----
html = '''
<link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/leaflet/1.9.4/leaflet.css"/>
<link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/leaflet.draw/1.0.4/leaflet.draw.css"/>
<style>.leaflet-image-layer{image-rendering:pixelated;}</style>
<div>
  <div style="margin-bottom:6px;">
    <button id="zDraw">Draw box</button>
    <button id="zClear">Clear box</button>
    <button id="zSend" style="font-weight:bold;">Use this box \u2014 send to Python</button>
    <button id="zBack">Back one level</button>
    <button id="zReset">Reset to original</button>
    <span id="zLabel" style="margin-left:10px;"></span>
  </div>
  <div id="zMap" style="width:100%;height:600px;border:1px solid #ccc;"></div>
  <div style="margin-top:6px;font:13px sans-serif;">
    __LO__ px <span style="display:inline-block;width:220px;height:12px;vertical-align:middle;background:linear-gradient(to right, __GRAD__);"></span> __HI__ px
    (log scale; green camera = lowest error)
  </div>
</div>
<script src="https://cdnjs.cloudflare.com/ajax/libs/leaflet/1.9.4/leaflet.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/leaflet.draw/1.0.4/leaflet.draw.js"></script>
<script>
(function() {
  const bounds = __BOUNDS__;
  const map = L.map('zMap').fitBounds(bounds);
  L.tileLayer('https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
              {maxNativeZoom: 18, maxZoom: 20, attribution: 'Esri World Imagery'}).addTo(map);
  L.imageOverlay('data:image/png;base64,__PNG__', bounds, {opacity: __OPACITY__}).addTo(map);
  __LANDMARKS__.forEach((p, i) => L.circleMarker(p, {radius: 4, color: 'black', fillColor: 'white', fillOpacity: 1})
                                   .bindTooltip('Landmark ' + (i+1)).addTo(map));
  L.marker(__BEST__).bindTooltip('Best: __BESTERR__ px').addTo(map);

  let box = null;
  const label = t => document.getElementById('zLabel').innerText = t;
  map.on(L.Draw.Event.CREATED, e => {
    if (box) map.removeLayer(box);
    box = e.layer; box.addTo(map);
    label('Box set. Click "Use this box" to send it.');
  });
  document.getElementById('zDraw').onclick = () => {
    new L.Draw.Rectangle(map, {showArea: false, shapeOptions: {color: '#00ff00', weight: 2}}).enable();
  };
  document.getElementById('zClear').onclick = () => { if (box) map.removeLayer(box); box = null; label(''); };
  document.getElementById('zSend').onclick = () => {
    if (!box) { label('Draw a box first.'); return; }
    const b = box.getBounds();
    google.colab.kernel.invokeFunction('notebook.set_action',
      ['box', [b.getSouth(), b.getWest(), b.getNorth(), b.getEast()]], {});
    label('Box sent! Re-run this cell to refine it.');
  };
  document.getElementById('zBack').onclick = () => {
    google.colab.kernel.invokeFunction('notebook.set_action', ['back', null], {});
    label('Back requested. Re-run this cell.');
  };
  document.getElementById('zReset').onclick = () => {
    google.colab.kernel.invokeFunction('notebook.set_action', ['reset', null], {});
    label('Reset requested. Re-run this cell.');

  };
})();
</script>
'''
html = (html.replace('__BOUNDS__', json.dumps([list(sw), list(ne)]))
            .replace('__PNG__', png_b64).replace('__OPACITY__', str(OPACITY))
            .replace('__LANDMARKS__', json.dumps([[la, lo_] for la, lo_, _ in landmarks_lla]))
            .replace('__BEST__', json.dumps([blat, blon])).replace('__BESTERR__', f"{b['err']:.2f}")
            .replace('__LO__', f"{10**lo:.2f}").replace('__HI__', f"{10**hi:.2f}").replace('__GRAD__', grad))
display(HTML(html))

In [ ]:
#@title Best-fit line through low-error heatmap region
from scipy.optimize import brentq
from scipy.constants import foot

ERR_TOL   = 0.25     # keep cells with error <= best * (1 + ERR_TOL)
MAX_CELLS = 5000     # if more qualify, keep only the lowest-error ones

v = view
assert v.get('bh') is not None, "Refine at least once with the patched zoom cell first."
ny, nx = v['err'].shape
es = v['e_lo'] + v['res'] * (np.arange(nx) + 0.5)
ns = v['n_lo'] + v['res'] * (np.arange(ny) + 0.5)
Eg, Ng = np.meshgrid(es, ns)

err = v['err']
mask = np.isfinite(err) & (err <= np.nanmin(err) * (1 + ERR_TOL))
pts  = np.c_[Eg[mask], Ng[mask], v['bh'][mask]]          # ENU meters rel. to landmark 1
errs = err[mask]
if len(pts) > MAX_CELLS:
    keep = np.argsort(errs)[:MAX_CELLS]
    pts, errs = pts[keep], errs[keep]
w = 1.0 / errs**2                                         # lower error -> more weight

# weighted PCA line fit
E0_hm = (w[:, None] * pts).sum(0) / w.sum()
_, s, vt = np.linalg.svd((pts - E0_hm) * np.sqrt(w)[:, None], full_matrices=False)
direction_hm = vt[0]
if direction_hm[2] < 0:                                   # t increasing = going up
    direction_hm = -direction_hm
r2_hm = s[0]**2 / np.sum(s**2)

t_pts = (pts - E0_hm) @ direction_hm
perp = (pts - E0_hm) - np.outer(t_pts, direction_hm)
fit_residual_hm = np.sqrt(np.average(np.sum(perp**2, axis=1), weights=w))

def _alt_at_t(t):
    p = E0_hm + t * direction_hm
    return pm.enu2geodetic(p[0], p[1], p[2], lat0, lon0, alt0)[2]

def _find_t_for_alt(target_alt_m, t_search_radius=1e9):
    f = lambda t: _alt_at_t(t) - target_alt_m
    lo, hi = -1.0, 1.0
    while f(lo) * f(hi) > 0:
        lo *= 2; hi *= 2
        if abs(lo) > t_search_radius:
            raise RuntimeError("Line is nearly horizontal; can't reach that altitude.")
    return brentq(f, lo, hi)

alt_min_ft_hm, alt_max_ft_hm = -20000.0, 20000.0
t_alt_min_hm, t_alt_max_hm = sorted([_find_t_for_alt(alt_min_ft_hm * foot),
                                     _find_t_for_alt(alt_max_ft_hm * foot)])

print(f"Used {len(pts):,} low-error cells (best {np.nanmin(err):.2f} px, cutoff {np.nanmin(err)*(1+ERR_TOL):.2f} px)")
print(f"Line-fit quality: R^2 = {r2_hm:.4f}  (weighted RMS perpendicular residual = {fit_residual_hm:.3f} m)\n")
print("Heatmap line, in ENU meters (relative to landmark 1):")
print(f"  Point on line   E0 = ({E0_hm[0]:.3f}, {E0_hm[1]:.3f}, {E0_hm[2]:.3f})")
print(f"  Direction (unit) d = ({direction_hm[0]:.5f}, {direction_hm[1]:.5f}, {direction_hm[2]:.5f})\n")
print("Parametric equation:")
print(f"  East(t)  = {E0_hm[0]:.3f} + t * ({direction_hm[0]:.5f})")
print(f"  North(t) = {E0_hm[1]:.3f} + t * ({direction_hm[1]:.5f})")
print(f"  Up(t)    = {E0_hm[2]:.3f} + t * ({direction_hm[2]:.5f})")
print(f"t = {t_alt_min_hm:.3f} -> altitude {_alt_at_t(t_alt_min_hm)/foot:.0f} ft")
print(f"t = {t_alt_max_hm:.3f} -> altitude {_alt_at_t(t_alt_max_hm)/foot:.0f} ft")

# compare with the solvePnP focal-sweep line
# ang = np.degrees(np.arccos(np.clip(abs(direction @ direction_hm), 0, 1)))
# d_perp = np.linalg.norm((E0_hm - E0) - ((E0_hm - E0) @ direction) * direction)
# print(f"\nAngle between the two lines: {ang:.2f} deg | closest-approach offset of heatmap centroid from PnP line: {d_perp:.1f} m")

# Visualizations

In [ ]:
#@title Interactive color scale for the error heatmap
import numpy as np, pymap3d as pm, folium, matplotlib
import ipywidgets as W
from IPython.display import display, clear_output
from matplotlib.colors import Normalize, to_hex
from branca.colormap import LinearColormap

logerr = np.log10(grid)                       # cached from the search cell
vmin_all, vmax_all = np.nanmin(logerr), np.nanmax(logerr)
p95 = np.nanpercentile(logerr, 95)

sw = pm.enu2geodetic(-radius_m - RES_M/2, -radius_m - RES_M/2, 0, lat0, lon0, alt0)[:2]
ne = pm.enu2geodetic( radius_m + RES_M/2,  radius_m + RES_M/2, 0, lat0, lon0, alt0)[:2]

# ---- controls ----
rng = W.FloatRangeSlider(value=[vmin_all, p95], min=vmin_all, max=vmax_all, step=0.01,
                         description='log10(px)', continuous_update=False,
                         readout_format='.2f', layout=W.Layout(width='600px'))
opa = W.FloatSlider(value=0.6, min=0, max=1, step=0.05, description='Opacity',
                    continuous_update=False)
cmap_dd = W.Dropdown(options=['RdYlBu', 'Spectral', 'coolwarm', 'jet', 'turbo', 'viridis'],
                     value='RdYlBu', description='Colormap')
rev = W.Checkbox(value=False, description='Flip (low = blue)')
base = W.Dropdown(options=['Satellite', 'Streets'], value='Satellite', description='Basemap')
info = W.Label()
out = W.Output()

def render(*_):
    lo, hi = rng.value
    cmap = matplotlib.colormaps[cmap_dd.value]
    if rev.value:
        cmap = cmap.reversed()
    rgba = cmap(Normalize(lo, hi, clip=True)(logerr))
    rgba[np.isnan(grid)] = 0
    rgba = np.flipud(rgba)
    info.value = f'color range: {10**lo:.2f} px (red) to {10**hi:.2f} px (blue)' if not rev.value \
                 else f'color range: {10**lo:.2f} px (blue) to {10**hi:.2f} px (red)'

    m = folium.Map(location=[lat0, lon0], zoom_start=10, tiles=None)
    if base.value == 'Satellite':
        folium.TileLayer('https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
                         attr='Esri World Imagery').add_to(m)
    else:
        folium.TileLayer('https://basemaps.cartocdn.com/rastertiles/voyager/{z}/{x}/{y}{r}.png',
                         attr='&copy; OpenStreetMap contributors &copy; CARTO').add_to(m)
    folium.raster_layers.ImageOverlay(rgba, bounds=[list(sw), list(ne)], opacity=opa.value).add_to(m)
    for k, (la, lo_, _) in enumerate(landmarks_lla, 1):
        folium.CircleMarker([la, lo_], radius=4, color='black', fill=True, fill_color='white',
                            tooltip=f'Landmark {k}').add_to(m)
    folium.Marker([blat, blon], tooltip=f'Best: {best[i]:.2f} px',
                  icon=folium.Icon(color='green', icon='camera')).add_to(m)
    LinearColormap([to_hex(cmap(x)) for x in np.linspace(0, 1, 8)], vmin=10**lo, vmax=10**hi,
                   caption='RMS reprojection error (px, log scale)').add_to(m)
    with out:
        clear_output(wait=True)
        display(m)

for w in (rng, opa, cmap_dd, rev, base):
    w.observe(render, names='value')

display(W.VBox([rng, W.HBox([cmap_dd, rev, base]), opa, info, out]))
render()

In [ ]:
#@title Visualization 1: KML File, both lines (can be imported to Google Earth)
import numpy as np, pymap3d as pm
from scipy.constants import foot
from google.colab import files

def line_lla(E0_, d_, t_lo, t_hi, n=200):
    ts = np.linspace(t_lo, t_hi, n)
    return np.array([pm.enu2geodetic(*(E0_ + t * d_), lat0, lon0, alt0) for t in ts])

def kml_color(rgb):
    r, g, b = [int(x) for x in rgb]
    return f'ff{b:02x}{g:02x}{r:02x}'          # KML color format is aabbggrr, not rrggbb

def line_placemark(p0, p1, color, width):
    return f'''<Placemark>
  <Style><LineStyle><color>{color}</color><width>{width}</width></LineStyle></Style>
  <LineString>
    <altitudeMode>absolute</altitudeMode>
    <coordinates>{p0[1]:.7f},{p0[0]:.7f},{p0[2]:.2f} {p1[1]:.7f},{p1[0]:.7f},{p1[2]:.2f}</coordinates>
  </LineString>
</Placemark>'''

def point_placemark(name, lat, lon, alt):
    return f'''<Placemark>
  <name>{name}</name>
  <Point>
    <altitudeMode>absolute</altitudeMode>
    <coordinates>{lon:.8f},{lat:.8f},{alt:.2f}</coordinates>
  </Point>
</Placemark>'''

hm_lla  = line_lla(E0_hm, direction_hm, t_alt_min_hm, t_alt_max_hm)

gray, green = np.array([0x88]*3), np.array([0x1a, 0x9e, 0x1a])


kml = ['<?xml version="1.0" encoding="UTF-8"?>',
       '<kml xmlns="http://www.opengis.net/kml/2.2"><Document>',
       '<name>Observer position lines</name>']

# --- Line A: solvePnP focal sweep (gray -> green by altitude, as before) ---
# kml.append('<Folder><name>solvePnP focal-sweep line</name>')
# for i in range(len(pnp_lla) - 1):
#     frac = max((pnp_lla[i, 2] + pnp_lla[i+1, 2]) / 2 / max_alt, 0)
#     kml.append(line_placemark(pnp_lla[i], pnp_lla[i+1], kml_color(gray + frac * (green - gray)), 5))
# kml.append('</Folder>')

# --- Line B: heatmap best fit (solid orange, one placemark) ---
kml.append('<Folder><name>Heatmap best-fit line</name>')
coords = ' '.join(f'{lo:.7f},{la:.7f},{al:.2f}' for la, lo, al in hm_lla)
kml.append(f'''<Placemark>
  <name>Heatmap best-fit line</name>
  <Style><LineStyle><color>{kml_color((0xff, 0x88, 0x00))}</color><width>4</width></LineStyle></Style>
  <LineString><altitudeMode>absolute</altitudeMode><coordinates>{coords}</coordinates></LineString>
</Placemark>''')
for row, tag in [(hm_lla[0], 'low'), (hm_lla[-1], 'high')]:
    kml.append(point_placemark(f'Heatmap line {tag} end, alt {row[2]/foot:.0f} ft', *row))
best_lat, best_lon, _ = pm.enu2geodetic(view['best']['e'], view['best']['n'], 0, lat0, lon0, alt0)
best_alt = alt0 + view['best']['h'] if view['best']['h'] is not None else alt0
kml.append(point_placemark(f"Heatmap best cell ({view['best']['err']:.2f} px)", best_lat, best_lon, best_alt))
kml.append('</Folder>')

# --- landmarks + PnP best guess ---
kml.append('<Folder><name>Landmarks</name>')
for i, (lat, lon, altv) in enumerate(landmarks_lla, 1):
    kml.append(point_placemark(f'Landmark {i} (alt {altv:.0f} m)', lat, lon, altv))
kml.append('</Folder>')
try:
    kml.append(point_placemark('Predicted Observer Position', obs_lat, obs_lon, obs_alt))
except NameError:
    pass

kml.append('</Document></kml>')
with open('observer_lines.kml', 'w') as f:
    f.write('\n'.join(kml))
files.download('observer_lines.kml')

In [ ]:
#@title Visualization 2: Google Maps, both lines (requires API key)
!pip install gmplot -q
import gmplot, portpicker, threading, http.server, socketserver
from google.colab import output

def line_lla(E0_, d_, t_lo, t_hi, n=200):
    ts = np.linspace(t_lo, t_hi, n)
    return np.array([pm.enu2geodetic(*(E0_ + t * d_), lat0, lon0, alt0) for t in ts])

hm_lla  = line_lla(E0_hm, direction_hm, t_alt_min_hm, t_alt_max_hm)

all_lla = np.vstack([hm_lla])
gmap = gmplot.GoogleMapPlotter(all_lla[:, 0].mean(), all_lla[:, 1].mean(), 16, apikey=GOOGLE_MAPS_API_KEY)
gmap.map_type = "satellite"

# --- Line A: solvePnP focal sweep (altitude gradient, as before) ---
# lats, lons, alts = pnp_lla[:, 0], pnp_lla[:, 1], pnp_lla[:, 2]
# gray, green, red = np.array([0x88]*3), np.array([0x1a, 0x9e, 0x1a]), np.array([0x9e, 0x1a, 0x1a])
# max_alt, min_alt = max(alts.max(), 1e-6), min(alts.min(), -1e-6)
# for i in range(len(lats) - 1):
#     a = (alts[i] + alts[i+1]) / 2
#     rgb = gray + (a / max_alt) * (green - gray) if a >= 0 else gray + (a / min_alt) * (red - gray)
#    gmap.plot([lats[i], lats[i+1]], [lons[i], lons[i+1]], color='#%02x%02x%02x' % tuple(rgb.astype(int)), edge_width=6)

# --- Line B: heatmap best fit (solid orange) ---
gmap.plot(hm_lla[:, 0], hm_lla[:, 1], color='#ff8800', edge_width=4)
for row, tag in [(hm_lla[0], "low"), (hm_lla[-1], "high")]:
    gmap.marker(row[0], row[1], color='orange', title=f"Heatmap line {tag} end, alt {row[2]/foot:.0f} ft")
best_lat, best_lon, _ = pm.enu2geodetic(view['best']['e'], view['best']['n'], 0, lat0, lon0, alt0)
gmap.marker(best_lat, best_lon, color='yellow', title=f"Heatmap best cell ({view['best']['err']:.2f} px)")

# --- landmarks + PnP best guess ---
for i, (lat, lon, altv) in enumerate(landmarks_lla):
    gmap.marker(lat, lon, color='black', title=f"Landmark {i+1} (alt {altv:.0f} m)")
try:
    gmap.marker(obs_lat, obs_lon, color='red', title=f"Predicted Observer Position, alt {obs_alt:.0f} m")
except NameError:
    pass

gmap.draw("observer_line_googlemaps.html")

port = portpicker.pick_unused_port()
def server_thread():
    class QuietHandler(http.server.SimpleHTTPRequestHandler):
        def log_message(self, format, *args): return
    with socketserver.TCPServer(("", port), QuietHandler) as httpd:
        httpd.serve_forever()
threading.Thread(target=server_thread, daemon=True).start()
output.serve_kernel_port_as_iframe(port, path='/observer_line_googlemaps.html', width="100%", height="650")